# Enterprise Orchestration Lab — RL Training

Manual REINFORCE loop with environment-grounded rewards. Colab T4 GPU required.

> If numpy errors occur: **Runtime > Factory reset runtime** first.

---
## 1. Install

In [ ]:
numpy_ok = True
try:
    import numpy as np; from numpy._core.umath import isalpha
    print(f'numpy {np.__version__} OK')
except Exception:
    numpy_ok = False; import subprocess
    subprocess.check_call(['pip','install','-q','numpy==1.26.4'])
    print('numpy fixed. Click Runtime > Restart session, then re-run.')
if numpy_ok:
    import subprocess
    subprocess.check_call(['pip','install','-q','trl','peft','accelerate','bitsandbytes','datasets','matplotlib'])
    subprocess.check_call(['pip','install','-q','openenv-core'])
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import get_peft_model, LoraConfig
    print(f'torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
    print('All imports OK.')

---
## 2. Setup Environment

In [ ]:
import os, sys, json, re, random
if not os.path.exists('scaler-final'):
    !git clone https://github.com/redhatsam09/scaler-final.git
if not os.getcwd().endswith('scaler-final'): os.chdir('scaler-final')
!pip install -q -r requirements.txt
if '.' not in sys.path: sys.path.insert(0, '.')
from src.environment import DataCleaningEnv
from src.graders import EnterpriseOrchestrationGrader, MissingValuesGrader
from src.models import Action
print('Environment loaded OK.')

---
## 3. Reward Function

Key fix: fuzzy-match action_type so the model gets credit for close guesses, and always attempt env.step() with a fallback action.

In [ ]:
VALID_ACTIONS = ['analyze','impute','deduplicate','validate','report_findings','delegate',
                 'resolve_alert','reconcile_apps','oversight_review','inspect_actor',
                 'audit_records','request_policy_clarification']
VALID_SET = set(VALID_ACTIONS)

SYSTEM_PROMPT = '''You are an enterprise workflow orchestrator managing CRM, Billing, and Support systems.
Output a JSON object with exactly these fields:
{
  "action_type": "<one of: analyze, impute, deduplicate, validate, report_findings, delegate, resolve_alert, reconcile_apps, oversight_review, inspect_actor, audit_records, request_policy_clarification>",
  "target_columns": ["column_name"],
  "parameters": {},
  "reasoning": "explanation of why this action (15+ chars)"
}
Respond with ONLY the JSON object, no other text.'''

def build_prompt(obs):
    return f'''{SYSTEM_PROMPT}

Current state: {obs.natural_language_observation[:400]}
Columns: {obs.column_names[:8]}
Missing: {json.dumps(dict(list(obs.missing_values.items())[:4]))}
Step: {obs.step_count}, KPIs: {json.dumps(obs.kpi_snapshot)}

Your JSON action:'''

def fuzzy_match_action(action_type):
    """Map model output to closest valid action_type."""
    action_type = action_type.lower().strip()
    if action_type in VALID_SET:
        return action_type, 1.0
    # Common mappings
    mappings = {'analysis':'analyze', 'analyse':'analyze', 'query':'analyze', 'check':'analyze',
                'fill':'impute', 'fillna':'impute', 'fill_missing':'impute', 'clean':'impute',
                'dedup':'deduplicate', 'remove_duplicates':'deduplicate',
                'verify':'validate', 'check_validity':'validate', 'validation':'validate',
                'report':'report_findings', 'findings':'report_findings', 'summary':'report_findings',
                'assign':'delegate', 'escalate':'delegate', 'forward':'delegate',
                'resolve':'resolve_alert', 'fix_alert':'resolve_alert', 'alert':'resolve_alert',
                'reconcile':'reconcile_apps', 'sync':'reconcile_apps', 'merge':'reconcile_apps',
                'oversight':'oversight_review', 'review':'oversight_review', 'monitor':'oversight_review',
                'inspect':'inspect_actor', 'check_actor':'inspect_actor',
                'audit':'audit_records', 'log':'audit_records', 'investigate':'audit_records',
                'clarify':'request_policy_clarification', 'policy':'request_policy_clarification'}
    if action_type in mappings:
        return mappings[action_type], 0.8
    # Substring match
    for valid in VALID_ACTIONS:
        if valid in action_type or action_type in valid:
            return valid, 0.6
    return 'analyze', 0.2  # Default fallback

def compute_reward(completion):
    """Multi-level reward with fuzzy action matching and env.step()."""
    # Level 0: Find JSON
    match = re.search(r'\{.*\}', completion, re.DOTALL)
    if not match: return -1.0
    
    # Level 1: Parse JSON
    try:
        data = json.loads(match.group())
    except json.JSONDecodeError:
        return -0.5
    if not isinstance(data, dict): return -0.4
    
    # Level 2: Check keys (partial credit)
    keys = ['action_type','target_columns','parameters','reasoning']
    has_keys = sum(1 for k in keys if k in data)
    key_score = has_keys / 4.0  # 0.25 to 1.0
    if has_keys == 0: return -0.3
    
    # Level 3: Match action_type (fuzzy)
    raw_action = str(data.get('action_type', 'analyze'))
    matched_action, match_quality = fuzzy_match_action(raw_action)
    
    # Level 4: Reasoning quality
    reasoning = str(data.get('reasoning', ''))
    reasoning_score = min(0.15, len(reasoning) / 100.0 * 0.15)
    
    # Level 5: Run through environment
    try:
        env = DataCleaningEnv(seed=42)
        obs = env.reset(task_id='task_enterprise_orchestration', seed=random.randint(0, 9999))
        target_cols = data.get('target_columns', [])
        if not isinstance(target_cols, list): target_cols = []
        action = Action(
            action_type=matched_action,
            target_columns=target_cols,
            parameters=data.get('parameters', {}) if isinstance(data.get('parameters'), dict) else {},
            reasoning=reasoning if len(reasoning) >= 15 else 'Auto-generated reasoning for action',
        )
        _, rw, _, _ = env.step(action)
        env_reward = rw.value
    except Exception:
        env_reward = 0.0
    
    # Combine: key_score(0-1) + match_quality(0.2-1) + env_reward + reasoning
    total = 0.2 * key_score + 0.3 * match_quality + 0.35 * env_reward + reasoning_score
    return max(-1.0, min(1.0, total))

# Test all levels
print('Reward function validation:')
tests = [
    ('no json', 'hello world'),
    ('bad json', '{broken'),
    ('wrong action', json.dumps({'action_type':'query_db','target_columns':['id'],'parameters':{},'reasoning':'need to check the data'})),
    ('close action', json.dumps({'action_type':'analyse','target_columns':['id'],'parameters':{},'reasoning':'Check data quality first'})),
    ('valid action', json.dumps({'action_type':'analyze','target_columns':['account_id'],'parameters':{},'reasoning':'Profile the full dataset quality metrics before corrective actions'})),
]
for label, comp in tests:
    print(f'  {label:15s} -> {compute_reward(comp):+.4f}')

---
## 4. Load Model

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, quantization_config=bnb_config, device_map='auto')
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=16, lora_dropout=0, task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model.print_trainable_parameters()

---
## 5. Generate Prompts

In [ ]:
prompt_env = DataCleaningEnv(seed=42)
task_ids = ['task_enterprise_orchestration','task_missing_values','task_duplicate_handling','task_complex_validation']
prompts = []
for i in range(40):
    obs = prompt_env.reset(task_id=task_ids[i%4], seed=2000+i)
    prompts.append(build_prompt(obs))
print(f'{len(prompts)} prompts ready')

---
## 6. REINFORCE Training Loop

Each step: generate -> reward -> log-probs -> policy gradient. Prints completion preview for debugging.

In [ ]:
import time

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
NUM_STEPS = 30
loss_history, reward_history = [], []
model.train()

print('Step | Loss       | Reward  | Preview')
print('-----|------------|---------|--------')

for step in range(NUM_STEPS):
    prompt = prompts[step % len(prompts)]
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    prompt_len = enc['input_ids'].shape[1]
    
    with torch.no_grad():
        gen_out = model.generate(**enc, max_new_tokens=200, temperature=1.0, do_sample=True, top_p=0.95)
    comp_ids = gen_out[0][prompt_len:]
    completion = tokenizer.decode(comp_ids, skip_special_tokens=True)
    
    reward = compute_reward(completion)
    reward_history.append(reward)
    
    with torch.amp.autocast('cuda', dtype=torch.float16):
        outputs = model(gen_out)
        logits = outputs.logits[0, prompt_len-1:-1, :]
        log_probs = torch.nn.functional.log_softmax(logits.float(), dim=-1)
        token_log_probs = log_probs.gather(1, comp_ids.unsqueeze(1)).squeeze(1)
        mean_log_prob = token_log_probs.mean()
    
    baseline = sum(reward_history) / len(reward_history)
    advantage = reward - baseline
    loss = -advantage * mean_log_prob
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    loss_history.append(loss.item())
    preview = completion[:80].replace('\n',' ')
    print(f'{step+1:4d} | {loss.item():+10.6f} | {reward:+.4f} | {preview}')

print(f'\nDone. Mean reward: {sum(reward_history)/len(reward_history):+.4f}')

---
## 7. Training Curves

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib
matplotlib.rcParams.update({'font.size':11,'figure.facecolor':'#0f172a','axes.facecolor':'#1e293b',
    'axes.edgecolor':'#334155','text.color':'#f8fafc','axes.labelcolor':'#94a3b8',
    'xtick.color':'#64748b','ytick.color':'#64748b'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(1,len(loss_history)+1), loss_history, color='#38bdf8', lw=2, marker='o', ms=3)
axes[0].fill_between(range(1,len(loss_history)+1), loss_history, alpha=0.15, color='#38bdf8')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].set_title('REINFORCE Training Loss', color='#38bdf8', fontweight='bold')
axes[0].grid(True, alpha=0.2, color='#475569')

axes[1].plot(range(1,len(reward_history)+1), reward_history, color='#22c55e', lw=2, marker='o', ms=3)
axes[1].fill_between(range(1,len(reward_history)+1), reward_history, alpha=0.15, color='#22c55e')
if len(reward_history) > 3:
    z = np.polyfit(range(len(reward_history)), reward_history, 1)
    axes[1].plot(range(1,len(reward_history)+1), np.poly1d(z)(range(len(reward_history))),
                 '--', color='#f59e0b', lw=1.5, label=f'Trend (slope={z[0]:.4f})')
    axes[1].legend(facecolor='#1e293b', edgecolor='#334155')
axes[1].axhline(y=0, color='#64748b', ls='--', alpha=0.4)
axes[1].set_xlabel('Step'); axes[1].set_ylabel('Reward')
axes[1].set_title('Per-Step Reward', color='#22c55e', fontweight='bold')
axes[1].grid(True, alpha=0.2, color='#475569')
plt.tight_layout()
plt.savefig('artifacts/training_curves.svg', facecolor='#0f172a')
plt.savefig('artifacts/training_curves.png', facecolor='#0f172a', dpi=150)
plt.show()

---
## 8. Evaluate

In [ ]:
eval_rewards = []
model.eval()
for i in range(12):
    obs = DataCleaningEnv(seed=99).reset(task_id=task_ids[i%4], seed=5000+i)
    prompt = build_prompt(obs)
    enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, temperature=0.7, do_sample=True)
    comp = tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    r = compute_reward(comp)
    eval_rewards.append(r)
    preview = comp[:60].replace('\n',' ')
    print(f'Episode {i+1:2d} | {task_ids[i%4]:36s} | reward={r:+.4f} | {preview}')

mean_r = sum(eval_rewards)/len(eval_rewards)
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(range(1,13), eval_rewards, color=['#22c55e' if r>0 else '#ef4444' for r in eval_rewards])
ax.axhline(y=0, color='#64748b', ls='--', alpha=0.4)
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.set_title(f'Eval Rewards (mean={mean_r:+.4f})', color='#38bdf8', fontweight='bold')
ax.grid(True, alpha=0.2, color='#475569')
plt.tight_layout()
plt.savefig('artifacts/eval_results.svg', facecolor='#0f172a')
plt.show()

---
## 9. Save Metrics

In [ ]:
from pathlib import Path
metrics = {'model': MODEL_NAME, 'method': 'REINFORCE', 'steps': NUM_STEPS,
    'loss_history': [round(l,6) for l in loss_history],
    'reward_history': [round(r,4) for r in reward_history],
    'eval_mean': round(mean_r,4), 'eval_per_episode': [round(r,4) for r in eval_rewards]}
Path('artifacts').mkdir(exist_ok=True)
Path('artifacts/grpo_training_metrics.json').write_text(json.dumps(metrics, indent=2))
print(json.dumps(metrics, indent=2))